# Ridge Regression (L2 Regularization) - Starter Notebook
Ridge adds an L2 penalty to OLS to shrink coefficients and reduce overfitting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

In [ ]:
np.random.seed(42)
n = 80
X = np.sort(np.random.uniform(0, 1, n))[:, None]
y = np.sin(2 * np.pi * X.ravel()) + np.random.randn(n) * 0.3
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Fit OLS and Ridge at degree-9 polynomial
degree = 9
alphas = [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]

X_plot = np.linspace(0, 1, 200)[:, None]
poly = PolynomialFeatures(degree)
scaler = StandardScaler()

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
axes = axes.flatten()

for ax, alpha in zip(axes, alphas):
    model = make_pipeline(PolynomialFeatures(degree), StandardScaler(),
                          Ridge(alpha=alpha) if alpha > 0 else LinearRegression())
    model.fit(X_train, y_train)
    y_pred = model.predict(X_plot)
    train_mse = mean_squared_error(y_train, model.predict(X_train))
    test_mse  = mean_squared_error(y_test,  model.predict(X_test))
    ax.scatter(X_train, y_train, color='gray', s=15, alpha=0.7)
    ax.plot(X_plot, y_pred, color='steelblue')
    ax.plot(X_plot, np.sin(2*np.pi*X_plot), 'r--', linewidth=1, label='True')
    label = 'OLS' if alpha == 0 else f'Ridge α={alpha}'
    ax.set_title(f'{label}\nTrain MSE={train_mse:.3f}, Test MSE={test_mse:.3f}', fontsize=8)
    ax.set_ylim(-2.5, 2.5)

plt.suptitle(f'Ridge Regression (degree={degree}): Effect of α', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Coefficient shrinkage path
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
alphas_path = np.logspace(-4, 4, 100)
Xp = StandardScaler().fit_transform(PolynomialFeatures(degree).fit_transform(X_train))
coefs = [Ridge(alpha=a).fit(Xp, y_train).coef_ for a in alphas_path]

plt.figure(figsize=(8, 4))
plt.semilogx(alphas_path, coefs)
plt.xlabel('Alpha (log scale)')
plt.ylabel('Coefficient value')
plt.title('Ridge Coefficient Shrinkage Path')
plt.axvline(1.0, color='red', linestyle='--', label='α=1')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validated alpha selection
cv_scores = []
for alpha in alphas_path:
    model = make_pipeline(PolynomialFeatures(degree), StandardScaler(), Ridge(alpha=alpha))
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
    cv_scores.append(-scores.mean())

best_alpha = alphas_path[np.argmin(cv_scores)]
print(f'Best alpha (5-fold CV): {best_alpha:.4f}')

plt.figure(figsize=(7, 4))
plt.semilogx(alphas_path, cv_scores, color='darkorange')
plt.axvline(best_alpha, color='red', linestyle='--', label=f'Best α={best_alpha:.4f}')
plt.xlabel('Alpha')
plt.ylabel('CV MSE')
plt.title('Ridge: Cross-Validation Error vs Alpha')
plt.legend()
plt.tight_layout()
plt.show()